### IMPORTS

In [1]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
scaler = StandardScaler()

### DATASETS

In [2]:
auto_mpg = pd.read_csv("auto_mpg_cleaned.csv")
auto_mpg = auto_mpg.dropna()
house_price = pd.read_csv('housing.csv')
house_price = house_price.dropna()
house_price = pd.get_dummies(house_price, dtype = int, drop_first = True)
insurance = pd.read_csv('insurance_cat2num.csv')

### CLASS SETUP

In [3]:
#One Hidden Layer Neural Network
        
'''======================================================================================'''

class OneHiddenLayerNNLeakyRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(OneHiddenLayerNNLeakyRelu, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # Input to hidden layer
        self.leaky_relu = nn.LeakyReLU()              # Activation function
        self.fc2 = nn.Linear(hidden_size, output_size) # Hidden to output layer
        self.batch_size = 32
        self.patience = 25
        self.min_delta = 1e-4
        
    def forward(self, x):
        out = self.fc1(x)   # Linear transformation
        out = self.leaky_relu(out) # Non-linearity
        out = self.fc2(out)  # Linear transformation to output
        return out
    def trainLeakyRelu(self, X_train, y_train, max_epochs=200, learning_rate=0.01):
        self.train()  # Set the model to training mode
        criterion = nn.MSELoss()  # Mean Squared Error Loss
        optimizer = torch.optim.SGD(self.parameters(), lr=learning_rate)  # Stochastic Gradient Descent

        batch_size = self.batch_size

        dataset = TensorDataset(X_train, y_train)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(max_epochs):
            epoch_loss = 0.0
            
            for batch_X, batch_y in dataloader:
            
                outputs = self(batch_X)  # Forward pass
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                epoch_loss+=loss.item()
            avg_loss = epoch_loss / len(dataloader)
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}/{max_epochs} | Avg Loss: {avg_loss:.4f}")

            if best_loss - avg_loss > self.min_delta:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= self.patience:
                print(f"\nEarly stopping triggered at Epoch {epoch+1}!")
                print(f"Loss hasn't improved by more than {self.min_delta} for {self.patience} consecutive epochs.")
                break
            
        print("Training Complete!")
    def testLeakyRelu(self, X_test, y_test):
        self.eval()  # Set the model to evaluation mode
        loss_fn = nn.MSELoss()
        with torch.no_grad():  # No need to compute gradients during testing
            outputs = self(X_test)  # Forward pass
            test_loss = loss_fn(outputs, y_test)
        print(f"Test Loss: {test_loss.item():.4f}")
        self.train()   
        return(outputs, test_loss.item())

### QOF VARIABLES

In [4]:
def get_qof(y_actual, y_pred, k, mod=None):
    """
    Calculates and compiles a comprehensive set of Quality of Fit (QoF) metrics 
    for a regression model.
    
    This function attempts to extract pre-calculated metrics from a fitted model 
    object (e.g., a Statsmodels result instance) if provided. If the object is not 
    provided or lacks specific metrics, it computes them manually using NumPy.

    Args:
        y_actual (array-like): The actual/true target values.
        y_pred (array-like): The predicted values generated by the model.
        k (int): The number of predictor variables (features) used in the model.
        mod (object, optional): A fitted model object (e.g., from Statsmodels) 
                                that may contain pre-calculated metrics. Defaults to None.

    Returns:
        list: A list containing 15 QoF metrics in the following specific order:
              [R^2, Adj R^2, SST, SSE, SDE, MSE, RMSE, MAE, sMAPE, 
               Number of Observations, DF Model, DF Residuals, F-statistic, AIC, BIC]
    """
    # ==========================================
    # --- Basic Parameters ---
    # ==========================================
    # Get the number of observations (m) in the original dataset
    m = len(y_actual)  

    # ==========================================
    # --- Sum of Squares ---
    # ==========================================
    # Calculate SSE (Sum of Squared Errors)
    sse = getattr(mod, "ssr", np.sum((y_actual - y_pred)**2))

    # Calculate SST (Total Sum of Squares)
    sst = getattr(mod, "centered_tss", np.sum((y_actual - np.mean(y_actual))**2))

    # ==========================================
    # --- R-Squared Metrics ---
    # ==========================================
    # Compute R-squared (Coefficient of Determination)
    r_squared = getattr(mod, "rsquared", 1 - (sse / sst))

    # Calculate Adjusted R-squared (penalizes for adding non-useful predictors)
    adj_r_squared = getattr(mod, "rsquared_adj", 1 - (1 - r_squared) * (m - 1) / (m - k))

    # ==========================================
    # --- Error Metrics ---
    # ==========================================
    # Standard Deviation of the Errors (SDE) / Residual Standard Error
    sde = np.sqrt(getattr(mod, "mse_resid", sse / (m - k)))

    # Calculate Mean Squared Error (MSE)
    mse = getattr(mod, "mse_resid", sse / m)

    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(getattr(mod, "mse_resid", mse))

    # Mean Absolute Error (MAE)
    mae = getattr(mod, "mae", np.mean(np.abs(y_actual - y_pred)))

    # Symmetric Mean Absolute Percentage Error (sMAPE)
    smape = getattr(mod, "smape", np.mean(2 * np.abs(y_actual - y_pred) / (np.abs(y_actual) + np.abs(y_pred))) * 100)

    # ==========================================
    # --- Statistical Tests & Degrees of Freedom ---
    # ==========================================
    # Calculate F-statistic for overall model significance
    f_stat = getattr(mod, "fvalue", ((sst - sse) / k) / (sse / (m - k)))

    # Get the degrees of freedom for the model (typically equivalent to k)
    df_model = getattr(mod, "df_model", k)

    # Get the degrees of freedom for the residuals (typically m - k)
    df_resid = getattr(mod, "df_resid", m - k)

    # ==========================================
    # --- Information Criteria ---
    # ==========================================
    # Calculate Akaike Information Criterion (AIC)
    aic = getattr(mod, "aic", m * np.log(mse) + 2 * k)

    # Calculate Bayesian Information Criterion (BIC)
    bic = getattr(mod, "bic", m * np.log(mse) + k * np.log(m))

    # ==========================================
    # --- Compile Results ---
    # ==========================================
    # Initialize a list to store exactly 15 elements
    qof = list(range(15))

    qof[0]  = r_squared
    qof[1]  = adj_r_squared
    qof[2]  = sst
    qof[3]  = sse
    qof[4]  = sde
    qof[5]  = mse
    qof[6]  = rmse
    qof[7]  = mae
    qof[8]  = smape
    qof[9]  = m
    qof[10] = df_model
    qof[11] = df_resid
    qof[12] = f_stat
    qof[13] = aic
    qof[14] = bic

    return qof

## IN-SAMPLE

### AUTOMPG

In [5]:
X = auto_mpg.drop('mpg', axis = 1)
y = auto_mpg[['mpg']]
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y.to_numpy(), dtype=torch.float32)

model_LR = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_LR.trainLeakyRelu(X_tensor, y_tensor)
(IS_predictions, IS_loss) = model_LR.testLeakyRelu(X_tensor, y_tensor)

y_numpy = y.to_numpy()
preds_IS_numpy = IS_predictions.cpu().numpy()
k = X_scaled.shape[1]
qof_IS = get_qof(y_numpy, preds_IS_numpy, k)
print(qof_IS)

Epoch 10/200 | Avg Loss: 8.7121
Epoch 20/200 | Avg Loss: 7.0923
Epoch 30/200 | Avg Loss: 6.8241
Epoch 40/200 | Avg Loss: 7.9784
Epoch 50/200 | Avg Loss: 6.7311
Epoch 60/200 | Avg Loss: 7.7649
Epoch 70/200 | Avg Loss: 6.2951
Epoch 80/200 | Avg Loss: 6.3352
Epoch 90/200 | Avg Loss: 5.8034
Epoch 100/200 | Avg Loss: 6.0208
Epoch 110/200 | Avg Loss: 5.4661
Epoch 120/200 | Avg Loss: 6.3706

Early stopping triggered at Epoch 127!
Loss hasn't improved by more than 0.0001 for 25 consecutive epochs.
Training Complete!
Test Loss: 5.5100
[np.float64(0.9093189006428791), np.float64(0.9079056887048461), np.float64(23818.99346938776), np.float64(2159.9325133841644), np.float64(2.368589099121743), np.float64(5.510031921898379), np.float64(2.3473457184442132), np.float64(1.713807361466544), np.float64(7.173026823589364), 392, 7, 385, np.float64(551.5210985521763), np.float64(682.9756033056409), np.float64(710.7744361841742)]


### HOUSE PRICES

In [6]:
X = house_price.drop('median_house_value', axis = 1)
y = house_price[['median_house_value']]
X_scaled = scaler.fit_transform(X)
y_scaled = y / 100000
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled.to_numpy(), dtype=torch.float32)

model_LR = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_LR.trainLeakyRelu(X_tensor, y_tensor)
(IS_predictions, IS_loss) = model_LR.testLeakyRelu(X_tensor, y_tensor)

y_numpy = y_scaled.to_numpy()
preds_IS_numpy = IS_predictions.cpu().numpy()
k = X_scaled.shape[1]
qof_IS = get_qof(y_numpy, preds_IS_numpy, k)
print(qof_IS)

Epoch 10/200 | Avg Loss: 0.3449
Epoch 20/200 | Avg Loss: 0.3191
Epoch 30/200 | Avg Loss: 0.3044
Epoch 40/200 | Avg Loss: 0.2951
Epoch 50/200 | Avg Loss: 0.2888
Epoch 60/200 | Avg Loss: 0.2846
Epoch 70/200 | Avg Loss: 0.2809
Epoch 80/200 | Avg Loss: 0.2776
Epoch 90/200 | Avg Loss: 0.2749
Epoch 100/200 | Avg Loss: 0.2720
Epoch 110/200 | Avg Loss: 0.2713
Epoch 120/200 | Avg Loss: 0.2688
Epoch 130/200 | Avg Loss: 0.2666
Epoch 140/200 | Avg Loss: 0.2656
Epoch 150/200 | Avg Loss: 0.2646
Epoch 160/200 | Avg Loss: 0.2624
Epoch 170/200 | Avg Loss: 0.2628
Epoch 180/200 | Avg Loss: 0.2617
Epoch 190/200 | Avg Loss: 0.2599
Epoch 200/200 | Avg Loss: 0.2605
Training Complete!
Test Loss: 0.2603
[np.float64(0.80462138764269), np.float64(0.8045161447683973), np.float64(27226.44346489301), np.float64(5319.464743595549), np.float64(0.5103821355355355), np.float64(0.26033694237730876), np.float64(0.5102322435688563), np.float64(0.35159880526902515), np.float64(18.030760998409217), 20433, 12, 20421, np.floa

### INSURANCE

In [7]:
X = insurance.drop('charges', axis = 1)
y = insurance[['charges']]
X_scaled = scaler.fit_transform(X)
y_scaled = y / 6000
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled.to_numpy(), dtype=torch.float32)

model_LR = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_LR.trainLeakyRelu(X_tensor, y_tensor)
(IS_predictions, IS_loss) = model_LR.testLeakyRelu(X_tensor, y_tensor)

y_numpy = y_scaled.to_numpy()
preds_IS_numpy = IS_predictions.cpu().numpy()
k = X_scaled.shape[1]
qof_IS = get_qof(y_numpy, preds_IS_numpy, k)
print(qof_IS)

Epoch 10/200 | Avg Loss: 0.6931
Epoch 20/200 | Avg Loss: 0.6234
Epoch 30/200 | Avg Loss: 0.6053
Epoch 40/200 | Avg Loss: 0.5959
Epoch 50/200 | Avg Loss: 0.5819
Epoch 60/200 | Avg Loss: 0.5757
Epoch 70/200 | Avg Loss: 0.5678
Epoch 80/200 | Avg Loss: 0.5595
Epoch 90/200 | Avg Loss: 0.5581
Epoch 100/200 | Avg Loss: 0.5557
Epoch 110/200 | Avg Loss: 0.5437
Epoch 120/200 | Avg Loss: 0.5412
Epoch 130/200 | Avg Loss: 0.5329
Epoch 140/200 | Avg Loss: 0.5289
Epoch 150/200 | Avg Loss: 0.5209
Epoch 160/200 | Avg Loss: 0.5220
Epoch 170/200 | Avg Loss: 0.5172
Epoch 180/200 | Avg Loss: 0.5151
Epoch 190/200 | Avg Loss: 0.5125
Epoch 200/200 | Avg Loss: 0.5093
Training Complete!
Test Loss: 0.5027
[np.float64(0.8764981981171854), np.float64(0.8757547711683047), np.float64(5446.506154676864), np.float64(672.6533240684321), np.float64(0.7114316070986414), np.float64(0.5027304365234918), np.float64(0.7090348626996361), np.float64(0.456103124417465), np.float64(26.377915260239522), 1338, 9, 1329, np.float64(

## TRAIN TEST SPLIT

### AUTOMPG

In [8]:
X = auto_mpg.drop('mpg', axis = 1)
y = auto_mpg[['mpg']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32)
model_LR = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_LR.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_LR.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 8.1732
Epoch 20/200 | Avg Loss: 11.1438
Epoch 30/200 | Avg Loss: 6.7494
Epoch 40/200 | Avg Loss: 7.1860
Epoch 50/200 | Avg Loss: 8.4925

Early stopping triggered at Epoch 55!
Loss hasn't improved by more than 0.0001 for 25 consecutive epochs.
Training Complete!
Test Loss: 6.3147
[np.float64(0.8762807142754098), np.float64(0.8659707737983606), np.float64(4032.206075949367), np.float64(498.86165561080855), np.float64(2.6322298732644294), np.float64(6.31470450140264), np.float64(2.512907579160571), np.float64(1.8028394868102258), np.float64(7.7442567273983745), 79, 7, 72, np.float64(72.85180324418133), np.float64(159.58759596486468), np.float64(176.17373093213382)]


### HOUSE PRICES

In [ ]:
X = house_price.drop('median_house_value', axis = 1)
y = house_price[['median_house_value']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = y_train / 100000
y_test_scaled = y_test / 100000
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled.to_numpy(), dtype=torch.float32)
model_LR = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_LR.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_LR.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test_scaled.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 0.3480
Epoch 20/200 | Avg Loss: 0.3251
Epoch 30/200 | Avg Loss: 0.3129
Epoch 40/200 | Avg Loss: 0.3040
Epoch 50/200 | Avg Loss: 0.2969
Epoch 60/200 | Avg Loss: 0.2898
Epoch 70/200 | Avg Loss: 0.2828
Epoch 80/200 | Avg Loss: 0.2793
Epoch 90/200 | Avg Loss: 0.2761
Epoch 100/200 | Avg Loss: 0.2732
Epoch 110/200 | Avg Loss: 0.2705
Epoch 120/200 | Avg Loss: 0.2702
Epoch 130/200 | Avg Loss: 0.2672
Epoch 140/200 | Avg Loss: 0.2651


### INSURANCE

In [ ]:
X = insurance.drop('charges', axis = 1)
y = insurance[['charges']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = y_train / 6000
y_test_scaled = y_test / 6000
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled.to_numpy(), dtype=torch.float32)
model_LR = OneHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size=100, output_size=y.shape[1])
model_LR.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_LR.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test_scaled.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)